In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# =====================================================
# LIMIT CPU THREADS TO PREVENT KAGGLE MEMORY CRASH
# =====================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


# =====================================================
# IMPORT LIBRARIES
# =====================================================

import pandas as pd
import numpy as np
import gc

from threadpoolctl import threadpool_limits

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_csv(
    "/kaggle/input/datasets/madhusudhanpallela/preprocessed1/preprocessed_dataset.csv",
    low_memory=True
)

TARGET = "diseases"


# =====================================================
# REMOVE MISSING TARGETS
# =====================================================

df = df.dropna(subset=[TARGET])


# =====================================================
# DISPLAY ORIGINAL CLASS DISTRIBUTION
# =====================================================

print("======================================")
print("ORIGINAL CLASS DISTRIBUTION")
print("======================================")

print(df[TARGET].value_counts())


# =====================================================
# REMOVE RARE CLASSES
# MINIMUM 10 SAMPLES PER CLASS
# =====================================================

counts = df[TARGET].value_counts()

valid_classes = counts[counts >= 10].index

df = df[df[TARGET].isin(valid_classes)].copy()


print("\n======================================")
print("CLASS DISTRIBUTION AFTER FILTERING")
print("======================================")

print(df[TARGET].value_counts())


# =====================================================
# FEATURES AND TARGET
# =====================================================

X = df.drop(columns=[TARGET])

y = df[TARGET]


# =====================================================
# IDENTIFY COLUMN TYPES
# =====================================================

numeric_features = X.select_dtypes(
    include=[
        "int32",
        "int64",
        "float32",
        "float64"
    ]
).columns


categorical_features = X.select_dtypes(
    include=[
        "object",
        "category",
        "bool"
    ]
).columns


print("\n======================================")
print("DATASET INFORMATION")
print("======================================")

print("Total Samples:", len(X))
print("Total Features:", X.shape[1])
print("Numerical Features:", len(numeric_features))
print("Categorical Features:", len(categorical_features))
print("Number of Classes:", y.nunique())


# =====================================================
# NUMERICAL PREPROCESSING
# =====================================================

numeric_transformer = Pipeline([
    
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    )

])


# =====================================================
# CATEGORICAL PREPROCESSING
# =====================================================

categorical_transformer = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        )
    )

])


# =====================================================
# COMPLETE PREPROCESSOR
# =====================================================

preprocessor = ColumnTransformer([

    (
        "num",
        numeric_transformer,
        numeric_features
    ),

    (
        "cat",
        categorical_transformer,
        categorical_features
    )

])


# =====================================================
# TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)


print("\n======================================")
print("DATA SPLIT")
print("======================================")

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))


# =====================================================
# RANDOM FOREST
# =====================================================

rf = RandomForestClassifier(

    n_estimators=300,

    max_depth=None,

    min_samples_split=2,

    min_samples_leaf=1,

    max_features="sqrt",

    class_weight="balanced",

    random_state=42,

    n_jobs=1

)


# =====================================================
# EXTRA TREES
# =====================================================

extra = ExtraTreesClassifier(

    n_estimators=300,

    max_depth=None,

    min_samples_split=2,

    min_samples_leaf=1,

    max_features="sqrt",

    class_weight="balanced",

    random_state=42,

    n_jobs=1

)


# =====================================================
# LOGISTIC REGRESSION
# =====================================================

lr = LogisticRegression(

    max_iter=1000,

    class_weight="balanced",

    C=2,

    solver="lbfgs"

)


# =====================================================
# BASE ESTIMATORS
# =====================================================

estimators = [

    (
        "rf",
        rf
    ),

    (
        "extra",
        extra
    ),

    (
        "lr",
        lr
    )

]


# =====================================================
# STACKING CLASSIFIER
# =====================================================

stack = StackingClassifier(

    estimators=estimators,

    final_estimator=LogisticRegression(

        max_iter=1000,

        C=1

    ),

    cv=3,

    n_jobs=1

)


# =====================================================
# COMPLETE MACHINE LEARNING PIPELINE
# =====================================================

model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        stack
    )

])


# =====================================================
# TRAIN MODEL
# =====================================================

print("\n======================================")
print("TRAINING STARTED")
print("======================================")

# Restrict BLAS threads during training
with threadpool_limits(limits=1):

    model.fit(
        X_train,
        y_train
    )


print("\n======================================")
print("TRAINING COMPLETED")
print("======================================")


# =====================================================
# PREDICTION
# =====================================================

print("\nGenerating Predictions...")

y_pred = model.predict(X_test)


# =====================================================
# ACCURACY
# =====================================================

accuracy = accuracy_score(

    y_test,

    y_pred

)


print("\n======================================")
print("FINAL MODEL ACCURACY")
print("======================================")

print(
    "Accuracy:",
    accuracy * 100,
    "%"
)


# =====================================================
# CLASSIFICATION REPORT
# =====================================================

print("\n======================================")
print("CLASSIFICATION REPORT")
print("======================================")

print(

    classification_report(

        y_test,

        y_pred,

        zero_division=0

    )

)


# =====================================================
# CONFUSION MATRIX
# =====================================================

print("\n======================================")
print("CONFUSION MATRIX")
print("======================================")

print(

    confusion_matrix(

        y_test,

        y_pred

    )

)


# =====================================================
# MEMORY CLEANUP
# =====================================================

gc.collect()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

# ===========================
# CONFIGURATION
# ===========================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

TRAIN_DIR = "/kaggle/input/datasets/youssefmohmmed/human-skin-diseases-image/SkinDisease/train"
VAL_DIR = "/kaggle/input/datasets/youssefmohmmed/human-skin-diseases-image/SkinDisease/valid"
TEST_DIR = "/kaggle/input/datasets/youssefmohmmed/human-skin-diseases-image/SkinDisease/test"

# ===========================
# LOAD DATASETS
# ===========================

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Number of classes:", NUM_CLASSES)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# ===========================
# DATA AUGMENTATION
# ===========================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.10),
])

# ===========================
# BUILD MODEL
# ===========================

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

# ===========================
# COMPILE MODEL
# ===========================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ===========================
# CALLBACKS
# ===========================

callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        verbose=1
    ),

    ModelCheckpoint(
        "best_efficientnet.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

# ===========================
# TRAIN
# ===========================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# ===========================
# FINE TUNING
# ===========================

base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

# ===========================
# TEST
# ===========================

test_loss, test_accuracy = model.evaluate(test_ds)

print("\nTest Accuracy:", test_accuracy)

# ===========================
# SAVE MODEL
# ===========================

model.save("efficientnet_skin_disease.keras")

print("Model saved successfully!")